<table style="width: 100%; border-collapse: collapse; border: none; background: #f0fdf4; border-left: 6px solid #10b981; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #064e3b; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Transformaciones Esenciales en Power Query ⚡
      </h1>
      <p style="margin: 6px 0 0 0; color: #10b981; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Visual Analytics and Critical Thinking
      </p>
      <p style="margin: 4px 0 0 0; color: #065f46; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #10b981; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        ⚡ Power Query — 01
      </span><br>
      <span style="color: #065f46; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #10b981; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Visual%20Analytics%20and%20Critical%20Thinking/Power%20Query/01_Transformaciones_Power_Query.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje ⚡

En este cuaderno dominamos las **transformaciones ETL más importantes** de Power Query, con su equivalente exacto en Pandas. Aprenderás a:

1. Combinar tablas con **Merge y Append** (JOIN y UNION en SQL).
2. **Pivotear y despivotar** tablas (reshape de datos).
3. **Limpiar datos**: nulos, tipos, texto, fechas.
4. **Agregar columnas** calculadas y condicionales.
5. **Unpivot** para convertir datos anchos en datos largos (tidy data).

In [ ]:
# ============================================================
# Setup: Datasets de trabajo para todas las transformaciones
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# Tabla principal: Ventas
df_ventas = pd.DataFrame({
    'id_orden': [f'ORD-{i:04d}' for i in range(1, 101)],
    'id_producto': np.random.choice(['P001', 'P002', 'P003', 'P004', 'P005'], 100),
    'id_cliente': np.random.choice([f'C{i:03d}' for i in range(1, 21)], 100),
    'fecha': pd.date_range('2023-01-01', periods=100, freq='3D'),
    'cantidad': np.random.randint(1, 20, 100),
    'precio_unitario': np.random.choice([45.99, 120.0, 890.0, 34.50, 250.0], 100),
    'descuento': np.random.choice([0, 0.05, 0.10, 0.15, 0.20], 100)
})
df_ventas['ventas_netas'] = (df_ventas['cantidad'] * df_ventas['precio_unitario'] *
                              (1 - df_ventas['descuento'])).round(2)

# Tabla de dimensión: Productos
df_productos = pd.DataFrame({
    'id_producto': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'nombre': ['Laptop', 'Mouse', 'Monitor', 'Teclado', 'Webcam'],
    'categoria': ['Tecnología', 'Periférico', 'Tecnología', 'Periférico', 'Periférico'],
    'costo': [600.0, 15.0, 250.0, 20.0, 45.0]
})

# Tabla de dimensión: Clientes
df_clientes = pd.DataFrame({
    'id_cliente': [f'C{i:03d}' for i in range(1, 21)],
    'nombre': [f'Cliente {chr(65+i)}' for i in range(20)],
    'ciudad': np.random.choice(['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Tunja'], 20),
    'segmento': np.random.choice(['Consumidor', 'Empresarial', 'Gobierno'], 20)
})

print("✅ Datasets listos:")
print(f"   df_ventas: {df_ventas.shape} — {df_ventas['ventas_netas'].sum():,.0f} en ventas")
print(f"   df_productos: {df_productos.shape}")
print(f"   df_clientes: {df_clientes.shape}")

---
## 1. Merge (JOIN): Combinar Tablas 🔗

### En Power Query:
**Inicio → Combinar consultas → Combinar (Merge)**

```m
// M Language — Inner Join
= Table.NestedJoin(
    Ventas, {"id_producto"},
    Productos, {"id_producto"},
    "ProductosUnidos",
    JoinKind.Inner
)
```

In [ ]:
# ============================================================
# MERGE: Tipos de JOIN en Pandas
# Equivalente a Table.NestedJoin en Power Query
# ============================================================

print("="*60)
print("MERGE (JOIN): Tipos disponibles")
print("="*60)

# 1. Inner Join (más común)
df_inner = pd.merge(df_ventas, df_productos, on='id_producto', how='inner')
print(f"\n1. INNER JOIN (M: JoinKind.Inner)")
print(f"   Filas resultado: {len(df_inner)} (solo coincidencias en ambas tablas)")

# 2. Left Join
df_left = pd.merge(df_ventas, df_productos, on='id_producto', how='left')
print(f"\n2. LEFT JOIN (M: JoinKind.LeftOuter)")
print(f"   Filas resultado: {len(df_left)} (todas las filas de ventas)")

# 3. Join múltiple (dos tablas de dimensión)
df_completo = (df_ventas
               .merge(df_productos, on='id_producto', how='left')
               .merge(df_clientes, on='id_cliente', how='left'))

print(f"\n3. JOIN MÚLTIPLE (ventas + productos + clientes)")
print(f"   Filas resultado: {len(df_completo)}")
print(f"   Columnas: {df_completo.shape[1]}")
print("\nVista previa del resultado:")
print(df_completo[['id_orden', 'nombre_x', 'ciudad', 'segmento', 'ventas_netas']].head(5).to_string())

print("\n📊 En Power Query:")
print("   1. Editar consulta de Ventas")
print("   2. Inicio → Combinar → Combinar consultas → seleccionar Productos")
print("   3. Seleccionar columna clave: id_producto en ambas tablas")
print("   4. Tipo de combinación: Externa izquierda")
print("   5. Expandir la columna anidada resultante")

---
##### 🛠️ Práctica 1: Merge con renombrado y Anti Join

En la Sección 1 notaste que al unir las tres tablas aparece `nombre_x`: `df_productos` y `df_clientes` tienen ambas una columna `nombre`. En Power Query lo resolverías renombrando antes de expandir la columna anidada. Hazlo con `df_ventas`, `df_productos` y `df_clientes`:

1. Renombra `nombre` a `producto` en `df_productos` y a `cliente` en `df_clientes` (`.rename(columns=...)`). Guarda los resultados en `productos_r` y `clientes_r`.
2. Une `df_ventas` con ambas (Left Join) y guárdalo en `df_enriq`. Agrega la columna `margen = ventas_netas - cantidad * costo`.
3. Calcula `margen_cat`: el margen total por `categoria` (`groupby`).
4. **Anti Join** (M: `JoinKind.LeftAnti`): guarda en `sin_compra_enero` los clientes de `df_clientes` que **no** compraron antes del `2023-02-01`. Pista: `merge(..., how='left', indicator=True)` y filtra `_merge == 'left_only'`.
5. Verifica con `assert`: `df_enriq` tiene las mismas filas que `df_ventas` y ninguna fila sin `producto` ni `cliente`; la suma de `margen_cat` coincide con la de `df_enriq['margen']`; y el conjunto de `id_cliente` de `sin_compra_enero` es exactamente el de los clientes sin ventas en enero.

In [ ]:
# Escribe tu código aquí

# productos_r = df_productos.rename(columns={'nombre': ...})
# clientes_r = df_clientes.rename(columns={'nombre': ...})
# df_enriq = (df_ventas
#             .merge(productos_r, on=..., how='left')
#             .merge(clientes_r, on=..., how='left'))
# df_enriq['margen'] = ...
# margen_cat = df_enriq.groupby(...)['margen'].sum()

# Anti join: clientes sin compras antes del 2023-02-01
# enero = df_ventas[df_ventas['fecha'] < '2023-02-01']
# sin_compra_enero = ...

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# Paso 1 (M: Table.RenameColumns): evitamos la colision de nombres
productos_r = df_productos.rename(columns={'nombre': 'producto'})
clientes_r = df_clientes.rename(columns={'nombre': 'cliente'})

# Paso 2 (M: NestedJoin + ExpandTableColumn): dos Left Join y margen
df_enriq = (df_ventas
            .merge(productos_r, on='id_producto', how='left')
            .merge(clientes_r, on='id_cliente', how='left'))
df_enriq['margen'] = df_enriq['ventas_netas'] - df_enriq['cantidad'] * df_enriq['costo']

# Paso 3: margen total por categoria
margen_cat = df_enriq.groupby('categoria')['margen'].sum()
print(margen_cat.round(2))

# Paso 4 (M: JoinKind.LeftAnti): clientes sin compras antes de febrero 2023
enero = df_ventas[df_ventas['fecha'] < '2023-02-01']
union = df_clientes.merge(enero[['id_cliente']].drop_duplicates(),
                          on='id_cliente', how='left', indicator=True)
sin_compra_enero = union[union['_merge'] == 'left_only'].drop(columns='_merge')
print(f"\nClientes sin compras en enero: {len(sin_compra_enero)} de {len(df_clientes)}")

assert len(df_enriq) == len(df_ventas)
assert df_enriq['producto'].notna().all() and df_enriq['cliente'].notna().all()
assert abs(margen_cat.sum() - df_enriq['margen'].sum()) < 1e-6
assert set(sin_compra_enero['id_cliente']) == set(df_clientes['id_cliente']) - set(enero['id_cliente'])
assert len(sin_compra_enero) > 0
```
</details>

---

---
## 2. Append: Apilar Tablas 📚

### En Power Query:
**Inicio → Combinar consultas → Anexar (Append)**

```m
// Equivalente a UNION ALL en SQL
= Table.Combine({Ventas2022, Ventas2023, Ventas2024})
```

In [ ]:
# APPEND: Apilar tablas verticalmente

# Simular datos de múltiples años
df_2022 = df_ventas[df_ventas['fecha'].dt.year == 2023].assign(año_origen=2022).head(30)
df_2023 = df_ventas[df_ventas['fecha'].dt.year == 2023].assign(año_origen=2023).tail(40)
df_2024 = df_ventas.sample(20, random_state=99).assign(año_origen=2024)

print("APPEND (UNION): Apilar múltiples tablas")
print(f"   df_2022: {len(df_2022)} filas")
print(f"   df_2023: {len(df_2023)} filas")
print(f"   df_2024: {len(df_2024)} filas")

# Pandas: pd.concat equivale a Table.Combine en M
df_todos = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)
print(f"\n   Resultado APPEND: {len(df_todos)} filas")
print("   Pandas: df_todos = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)")
print("   M:      = Table.Combine({Ventas2022, Ventas2023, Ventas2024})")

---
## 3. Pivot y Unpivot: Reshape de Datos 🔄

### Conceptos clave:
- **Pivotear** (wide format): convertir valores de una columna en nuevas columnas
- **Despivotar / Unpivot** (long/tidy format): convertir columnas en filas

El formato **tidy** (largo) es el estándar para visualización en Python, Power BI y Tableau.

In [ ]:
# ============================================================
# PIVOT y UNPIVOT: Reshape
# ============================================================

import pandas as pd
import numpy as np

# Datos en formato ANCHO (necesita unpivot para visualización)
df_ancho = pd.DataFrame({
    'Mes': ['Enero', 'Febrero', 'Marzo', 'Abril'],
    'Norte': [45000, 48000, 52000, 49000],
    'Sur': [32000, 35000, 38000, 34000],
    'Este': [28000, 30000, 33000, 31000],
    'Oeste': [38000, 41000, 44000, 40000]
})

print("DATOS EN FORMATO ANCHO (Wide Format):")
print(df_ancho.to_string(index=False))
print("\n⚠️  Formato ancho: difícil de usar en gráficos y Power BI/Tableau")

# UNPIVOT (M: Table.Unpivot)
df_largo = df_ancho.melt(
    id_vars=['Mes'],       # Columnas que NO se transforman
    var_name='Region',     # Nombre de la nueva columna de categorías
    value_name='Ventas'    # Nombre de la nueva columna de valores
)

print("\nDATOS EN FORMATO LARGO/TIDY (después de Unpivot):")
print(df_largo.to_string(index=False))
print(f"\n✅ Formato largo: {len(df_largo)} filas — ideal para gráficos grouped bar")

print("\nEquivalencias:")
print("   Pandas:        df.melt(id_vars=['Mes'], var_name='Region', value_name='Ventas')")
print("   M (PQ):        Table.Unpivot(Tabla, {\"Norte\", \"Sur\", \"Este\", \"Oeste\"}, \"Region\", \"Ventas\")")
print("   Power Query GUI: Seleccionar columnas de regiones → Transformar → Anular dinamización")

# PIVOT (el inverso)
print("\n" + "="*50)
df_pivot = df_largo.pivot_table(
    values='Ventas',
    index='Mes',
    columns='Region',
    aggfunc='sum'
).reset_index()
print("PIVOT (volver al formato ancho):")
print(df_pivot.to_string(index=False))
print("   Pandas: df.pivot_table(values='Ventas', index='Mes', columns='Region')")
print("   M:      Table.Pivot(Tabla, List.Distinct([Region]), \"Region\", \"Ventas\", List.Sum)")

---
##### 🛠️ Práctica 2: Append, Pivot y Unpivot en cadena

Con `df_ventas` practica tres operaciones seguidas: apilar (`Table.Combine`), pivotar (`Table.Pivot`) y despivotar (`Table.Unpivot`).

1. Crea `h1` (ventas con `fecha` en los meses 1 a 6) y `h2` (meses 7 a 12). A `h2` agrégale una columna `nota = 'segundo semestre'`. Apílalas con `pd.concat([h1, h2], ignore_index=True)` y guárdalo en `df_anexado`. Observa que, igual que en M, las columnas que no existen en una tabla se llenan con nulo (`null`).
2. Agrega la columna `mes` (`fecha.dt.month`) a `df_anexado` y crea `tabla_ancha`: ventas netas sumadas con `mes` en las filas y `id_producto` en las columnas (`pivot_table(..., fill_value=0)` y luego `reset_index()`).
3. Despivota `tabla_ancha` con `.melt(...)` para volver al formato largo (`mes`, `id_producto`, `ventas_netas`) y guárdalo en `tabla_larga`.
4. Verifica con `assert`: `df_anexado` tiene las mismas filas que `df_ventas` y tantos nulos en `nota` como filas tiene `h1`; la suma de `ventas_netas` de `tabla_larga` coincide con la de `df_ventas`; y `tabla_larga` tiene `5 *` el número de meses (5 productos).

In [ ]:
# Escribe tu código aquí

# h1 = df_ventas[df_ventas['fecha'].dt.month <= ...]
# h2 = df_ventas[df_ventas['fecha'].dt.month > ...].assign(nota=...)
# df_anexado = pd.concat([..., ...], ignore_index=True)
# df_anexado['mes'] = ...

# tabla_ancha = df_anexado.pivot_table(values=..., index=..., columns=..., aggfunc='sum', fill_value=0).reset_index()
# tabla_larga = tabla_ancha.melt(id_vars=..., var_name=..., value_name=...)

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# Paso 1 (M: Table.Combine): apilar dos tablas con columnas distintas
h1 = df_ventas[df_ventas['fecha'].dt.month <= 6]
h2 = df_ventas[df_ventas['fecha'].dt.month > 6].assign(nota='segundo semestre')
df_anexado = pd.concat([h1, h2], ignore_index=True)       # 'nota' queda null en h1
df_anexado['mes'] = df_anexado['fecha'].dt.month

# Paso 2 (M: Table.Pivot): mes en filas, producto en columnas
tabla_ancha = df_anexado.pivot_table(values='ventas_netas', index='mes',
                                     columns='id_producto', aggfunc='sum',
                                     fill_value=0).reset_index()
print(tabla_ancha.round(0).to_string(index=False))

# Paso 3 (M: Table.Unpivot): de vuelta al formato largo
tabla_larga = tabla_ancha.melt(id_vars='mes', var_name='id_producto', value_name='ventas_netas')
print(f"\nFormato largo: {tabla_larga.shape}")

assert len(df_anexado) == len(df_ventas)
assert df_anexado['nota'].isna().sum() == len(h1)
assert abs(tabla_larga['ventas_netas'].sum() - df_ventas['ventas_netas'].sum()) < 1e-6
assert len(tabla_larga) == 5 * tabla_ancha['mes'].nunique()
```
</details>

---

---
## 4. Limpieza de Datos: Nulos, Duplicados y Texto 🧹

| Operación | Power Query GUI | M Language | Pandas |
|---|---|---|---|
| **Eliminar nulos** | Inicio → Quitar filas → Quitar filas en blanco | `Table.SelectRows(t, each not List.AnyTrue(List.Transform(..., Value.Is(_, Null.Type))))` | `df.dropna()` |
| **Rellenar nulos** | Transformar → Reemplazar valores → null → 0 | `Table.ReplaceValue(t, null, 0, ...)` | `df.fillna(0)` |
| **Eliminar duplicados** | Inicio → Quitar filas → Quitar duplicados | `Table.Distinct(t)` | `df.drop_duplicates()` |
| **Recortar espacios** | Transformar → Formato → Recortar | `Text.Trim(texto)` | `df[col].str.strip()` |
| **Mayúsculas** | Transformar → Formato → Mayúsculas | `Text.Upper(texto)` | `df[col].str.upper()` |
| **Extraer texto** | Transformar → Extraer → Texto antes del delimitador | `Text.BeforeDelimiter(texto, "-")` | `df[col].str.split('-').str[0]` |

In [ ]:
# ============================================================
# Limpieza de datos con Pandas (equivalente a Power Query)
# ============================================================

# Dataset con problemas de calidad
df_sucio = pd.DataFrame({
    'nombre': ['  Juan García', 'María López', None, 'carlos MARTINEZ', 'Juan García', '  Pedro  '],
    'ventas': [1200.0, None, 800.0, 1500.0, 1200.0, 900.0],
    'fecha': ['2023-01-15', '2023/02/20', 'invalid_date', '2023-03-10', '2023-01-15', '2023-04-05'],
    'codigo': ['A-001-X', 'B-002-Y', 'C-003-Z', 'D-004-W', 'A-001-X', 'E-005-V']
})

print("DATOS SUCIOS:")
print(df_sucio.to_string())
print(f"\nProblemas detectados:")
print(f"  - Espacios en nombres: 'Juan García', 'Pedro'")
print(f"  - Capitalización inconsistente: 'carlos MARTINEZ'")
print(f"  - Nulos en 'nombre' y 'ventas'")
print(f"  - Fechas con formatos mixtos")
print(f"  - Duplicados: 'Juan García' aparece 2 veces")

print("\n" + "="*50)
print("LIMPIEZA (equivalente a Power Query):")

df_limpio = (
    df_sucio
    # 1. Recortar espacios y capitalizar → Text.Trim + Text.Proper en M
    .assign(nombre=lambda x: x['nombre'].str.strip().str.title())
    # 2. Rellenar nulos en ventas → Table.ReplaceValue en M
    .assign(ventas=lambda x: x['ventas'].fillna(x['ventas'].median()))
    # 3. Eliminar fila con nombre nulo → Table.SelectRows(..., each [nombre] <> null)
    .dropna(subset=['nombre'])
    # 4. Parsear fechas → Table.TransformColumnTypes en M
    .assign(fecha=lambda x: pd.to_datetime(x['fecha'], errors='coerce'))
    # 5. Extraer código base → Text.BeforeDelimiter en M
    .assign(codigo_base=lambda x: x['codigo'].str.split('-').str[:2].str.join('-'))
    # 6. Eliminar duplicados → Table.Distinct en M
    .drop_duplicates(subset=['nombre', 'ventas'])
    .reset_index(drop=True)
)

print("\nDATOS LIMPIOS:")
print(df_limpio.to_string())
print(f"\n✅ De {len(df_sucio)} filas → {len(df_limpio)} filas limpias")

---
## 5. Columnas Condicionales: IF en M vs np.where en Python 🔀

```m
// M Language: Columna condicional
= Table.AddColumn(Tabla, "Categoria_Venta",
    each if [Ventas] > 1000 then "Alta"
         else if [Ventas] > 500 then "Media"
         else "Baja"
)
```

In [ ]:
# Columna condicional: M (if...else) vs Pandas (np.where / pd.cut)

import numpy as np

df_cond = df_ventas[['id_orden', 'ventas_netas']].copy()

# Método 1: np.where (equivale a IF anidado simple)
df_cond['categoria_np'] = np.where(
    df_cond['ventas_netas'] > 1000, 'Alta',
    np.where(df_cond['ventas_netas'] > 500, 'Media', 'Baja')
)

# Método 2: pd.cut (más legible para múltiples rangos)
df_cond['categoria_cut'] = pd.cut(
    df_cond['ventas_netas'],
    bins=[0, 500, 1000, np.inf],
    labels=['Baja', 'Media', 'Alta']
)

print("COLUMNAS CONDICIONALES:")
print(df_cond.head(10).to_string())
print("\nDistribución de categorías:")
print(df_cond['categoria_np'].value_counts())

print("\n📊 En Power Query GUI:")
print("   Transformar → Agregar columna → Columna condicional")
print("   Definir las condiciones en la interfaz visual")
print("   Power Query genera el código M automáticamente")

---
##### 🛠️ Práctica 3: Limpia una tabla sucia y clasifícala

Pon todo junto con una tabla pequeña de facturas con problemas de calidad. Defínela en tu código:

```python
df_fact = pd.DataFrame({
    'cliente': ['  ana RUIZ', 'Luis Pérez', None, 'MARTA gil ', 'luis pérez'],
    'monto':   ['1,200.50', '850', '300', 'n/a', '850'],
    'orden':   ['FAC-2023-001', 'FAC-2023-002', 'FAC-2023-003', 'FAC-2023-004', 'FAC-2023-002'],
})
```

Escribe una cadena de pasos (como en la Sección 4) que produzca `df_fact_limpia`:

1. `cliente`: recortar espacios y poner formato título (M: `Text.Trim` y `Text.Proper`).
2. `monto`: quitar la coma de miles y convertir a número; los valores no numéricos (`'n/a'`) deben quedar nulos (`pd.to_numeric(..., errors='coerce')`).
3. Eliminar las filas con `cliente` o `monto` nulos (`dropna(subset=[...])`).
4. Crear `anio` con el año contenido en `orden` (M: `Text.BetweenDelimiters`), como entero.
5. Eliminar duplicados por `cliente` y `orden`.
6. Columna condicional `segmento` (M: `if [monto] >= 1000 then "Alto" else if [monto] >= 500 then "Medio" else "Bajo"`) con `np.select`.
7. Verifica con `assert`: quedan 2 filas, los clientes son `['Ana Ruiz', 'Luis Pérez']`, `anio` es 2023 en todas, y `segmento` es `['Alto', 'Medio']`.

In [ ]:
# Escribe tu código aquí

# df_fact = pd.DataFrame({
#     'cliente': ['  ana RUIZ', 'Luis Pérez', None, 'MARTA gil ', 'luis pérez'],
#     'monto':   ['1,200.50', '850', '300', 'n/a', '850'],
#     'orden':   ['FAC-2023-001', 'FAC-2023-002', 'FAC-2023-003', 'FAC-2023-004', 'FAC-2023-002'],
# })

# df_fact_limpia = (df_fact
#     .assign(cliente=lambda x: x['cliente'].str.strip().str.title(),
#             monto=lambda x: pd.to_numeric(x['monto'].str.replace(',', ''), errors=...))
#     .dropna(subset=[..., ...])
#     .assign(anio=lambda x: x['orden'].str.split('-').str[...].astype(int))
#     .drop_duplicates(subset=[..., ...])
#     .reset_index(drop=True))

# condiciones = [df_fact_limpia['monto'] >= ..., df_fact_limpia['monto'] >= ...]
# df_fact_limpia['segmento'] = np.select(condiciones, ['Alto', 'Medio'], default='Bajo')

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
df_fact = pd.DataFrame({
    'cliente': ['  ana RUIZ', 'Luis Pérez', None, 'MARTA gil ', 'luis pérez'],
    'monto':   ['1,200.50', '850', '300', 'n/a', '850'],
    'orden':   ['FAC-2023-001', 'FAC-2023-002', 'FAC-2023-003', 'FAC-2023-004', 'FAC-2023-002'],
})

df_fact_limpia = (df_fact
    # 1. Text.Trim + Text.Proper
    .assign(cliente=lambda x: x['cliente'].str.strip().str.title(),
            # 2. Text.Replace + Number.FromText (try ... otherwise null)
            monto=lambda x: pd.to_numeric(x['monto'].str.replace(',', ''), errors='coerce'))
    # 3. Table.SelectRows(..., each [cliente] <> null and [monto] <> null)
    .dropna(subset=['cliente', 'monto'])
    # 4. Text.BetweenDelimiters: 'FAC-2023-001' -> 2023
    .assign(anio=lambda x: x['orden'].str.split('-').str[1].astype(int))
    # 5. Table.Distinct sobre cliente y orden
    .drop_duplicates(subset=['cliente', 'orden'])
    .reset_index(drop=True))

# 6. Columna condicional (if ... else if ... else): np.select evalua en orden
condiciones = [df_fact_limpia['monto'] >= 1000, df_fact_limpia['monto'] >= 500]
df_fact_limpia['segmento'] = np.select(condiciones, ['Alto', 'Medio'], default='Bajo')

print(df_fact_limpia.to_string())

assert len(df_fact_limpia) == 2
assert df_fact_limpia['cliente'].tolist() == ['Ana Ruiz', 'Luis Pérez']
assert (df_fact_limpia['anio'] == 2023).all()
assert df_fact_limpia['segmento'].tolist() == ['Alto', 'Medio']
```
</details>

---

---
## 6. Resumen de Transformaciones ⚡

| Transformación | Power Query GUI | M Language | Pandas |
|---|---|---|---|
| **Combinar tablas** | Combinar → Combinar consultas | `Table.NestedJoin(...)` | `pd.merge(df1, df2, on=...)` |
| **Apilar tablas** | Combinar → Anexar consultas | `Table.Combine({t1, t2, t3})` | `pd.concat([df1, df2, df3])` |
| **Despivotar** | Transformar → Anular dinamización | `Table.Unpivot(t, cols, attr, val)` | `df.melt(id_vars=..., var_name=..., value_name=...)` |
| **Pivotar** | Transformar → Dinamizar columna | `Table.Pivot(t, cols, attr, val, agg)` | `df.pivot_table(values=..., index=..., columns=...)` |
| **Limpiar nulos** | Inicio → Quitar filas en blanco | `Table.SelectRows(t, each [col] <> null)` | `df.dropna()` / `df.fillna()` |
| **Eliminar duplicados** | Inicio → Quitar duplicados | `Table.Distinct(t)` | `df.drop_duplicates()` |
| **Columna calculada** | Agregar columna → Columna personalizada | `Table.AddColumn(t, "nom", each ...)` | `df['nueva'] = ...` |
| **Columna condicional** | Agregar columna → Columna condicional | `each if [col] > x then "A" else "B"` | `np.where(cond, 'A', 'B')` |

> 🚀 **Siguiente cuaderno:** [Power Query vs Pandas — Tabla Comparativa](02_Power_Query_vs_Pandas.ipynb)